In [ ]:
# --- HÜCRE 1: Kurulumlar (Unsloth ile Süper Hızlı Eğitim) ---
print("Kurulumlar yapılıyor... (1-2 dk)")
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes --quiet
!pip install pandas-datareader scikit-learn --quiet
# WandB'yi devre dışı bırakma (Önceki hatayı engeller)
import os
os.environ["WANDB_DISABLED"] = "true"
print("Kurulum tamamlandı. WandB devre dışı bırakıldı.")

Kurulumlar yapılıyor... (1-2 dk)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 22.9 MB/s eta 0:00:00
Kurulum tamamlandı. WandB devre

In [ ]:
# --- HÜCRE 2: Veri Mühendisliği (10x10 Çözünürlük - Plan B) ---
import pandas as pd
import numpy as np
import torch
from pandas_datareader import data as pdr
from datetime import datetime, timedelta
from sklearn.metrics import mean_squared_error
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# PLAN B: 10 Seviyeli Hassas Dil (Deciles)
PRICE_LABELS = [f"P_{i}" for i in range(10)]
VOLUME_LABELS = [f"V_{i}" for i in range(10)]

class FinwiseSymbolizer:
    def __init__(self, tickers, period="10y", n_quantiles=10): # <-- 10 yaptık
        self.tickers = tickers
        self.period = period
        self.n_quantiles = n_quantiles
        self.price_labels = PRICE_LABELS
        self.volume_labels = VOLUME_LABELS

    def fetch_data(self):
        print(f"Veri çekiliyor (Stooq)...")
        start_date = datetime.now() - timedelta(days=365*10)
        end_date = datetime.now()

        all_data = []
        for ticker in self.tickers:
            try:
                df = pdr.get_data_stooq(f"{ticker}.US", start=start_date, end=end_date)
                df = df.sort_index(ascending=True)
                df = df[['Close', 'Volume']].rename(columns={'Close': f'{ticker}_Close', 'Volume': f'{ticker}_Volume'})
                all_data.append(df)
            except Exception as e:
                print(f"{ticker} verisi çekilemedi, atlanıyor: {e}")

        if not all_data: return pd.DataFrame()
        return pd.concat(all_data, axis=1).dropna()

    def process(self, df):
        print(f"Veri işleniyor... ({self.n_quantiles}x{self.n_quantiles} Çözünürlük)")
        pct_df = pd.DataFrame(index=df.index)
        tokens_df = pd.DataFrame(index=df.index)

        tickers = set([c.split('_')[0] for c in df.columns])

        for t in tickers:
            p_col, v_col = f"{t}_Close", f"{t}_Volume"
            if p_col not in df.columns: continue

            pct_df[f"{t}_P_Change"] = df[p_col].pct_change()
            pct_df[f"{t}_V_Change"] = df[v_col].pct_change()

        pct_df = pct_df.replace([np.inf, -np.inf], np.nan).dropna()

        # Yüksek Çözünürlüklü Tokenize Etme
        for t in tickers:
            try:
                tokens_df[f"{t}_P_Token"] = pd.qcut(pct_df[f"{t}_P_Change"], self.n_quantiles, labels=self.price_labels, duplicates='drop')
                tokens_df[f"{t}_V_Token"] = pd.qcut(pct_df[f"{t}_V_Change"], self.n_quantiles, labels=self.volume_labels, duplicates='drop')
            except ValueError:
                # Yedek Çözüm: rank yöntemi
                print(f"Uyarı: {t} için qcut zorlanıyor, rank yöntemi kullanılıyor.")
                tokens_df[f"{t}_P_Token"] = pd.qcut(pct_df[f"{t}_P_Change"].rank(method='first'), self.n_quantiles, labels=self.price_labels)
                tokens_df[f"{t}_V_Token"] = pd.qcut(pct_df[f"{t}_V_Change"].rank(method='first'), self.n_quantiles, labels=self.volume_labels)

        combined_tokens = []
        for t in tickers:
            combined_tokens.append(tokens_df[f"{t}_P_Token"].astype(str) + "_" + tokens_df[f"{t}_V_Token"].astype(str))

        final_text = pd.DataFrame(combined_tokens).T.agg(' '.join, axis=1)
        return pct_df, tokens_df, final_text

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.9.0+cu128 with CUDA 1208 (you have 2.8.0+cu126)
    Python  3.10.19 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.9.0+cu128 with CUDA 1208 (you have 2.8.0+cu126)
    Python  3.10.19 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
!nvidia-smi

Wed Nov 19 17:21:00 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             26W /   70W |     102MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
# --- HÜCRE 3: Model Eğitimi (300 Adım - Plan B) ---

# 1. Veri Hazırlığı
# n_quantiles=10'u burada da kullan
symbolizer = FinwiseSymbolizer(tickers=["MSFT", "AAPL", "GOOGL", "AMZN", "NVDA"], period="10y", n_quantiles=10)
raw_df = symbolizer.fetch_data()

if raw_df.empty:
    raise ValueError("Veri çekilemedi! Lütfen Stooq bağlantısını kontrol edin.")

num_df, sym_df, text_series = symbolizer.process(raw_df)

target_ticker = "MSFT"
train_prompts = []

full_tokens = sym_df[f"{target_ticker}_P_Token"].astype(str) + "_" + sym_df[f"{target_ticker}_V_Token"].astype(str)
token_to_value = pd.concat([full_tokens, num_df[f"{target_ticker}_P_Change"]], axis=1).groupby(0).mean().to_dict()[f"{target_ticker}_P_Change"]

print(f"Veri seti hazırlanıyor... Toplam gün sayısı: {len(text_series)}")
context_window = 60
for i in range(context_window, len(text_series)):
    history = " ".join(text_series.iloc[i-context_window:i].values)
    target = full_tokens.iloc[i]

    prompt = f"Predict the next market token for {target_ticker} based on history:\n{history}\nResponse: {target}"
    train_prompts.append(prompt)

dataset = Dataset.from_dict({"text": train_prompts})
train_test_split = dataset.train_test_split(test_size=0.1)

# 2. Modeli Yükle (VRAM Optimizasyonu)
print("Llama-3 Modeli Yükleniyor (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 1024, # <-- KRİTİK DÜZELTME: 2048 YERİNE 1024 YAPILDI
    dtype = None,
    load_in_4bit = True,
)

# LoRA Adaptörlerini Ekle (Değişmedi)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# 3. Eğitimi Başlat
print("--- EĞİTİM BAŞLIYOR ---")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_test_split["train"],
    dataset_text_field = "text",
    max_seq_length = 1024, # Değişmedi
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1, # <-- KRİTİK: 2 YERİNE 1 YAPILDI (VRAM'i yarılar)
        gradient_accumulation_steps = 8, # <-- KRİTİK: 4 YERİNE 8 YAPILDI (Efektif Batch Size = 1 * 8 = 8, değişmedi)
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 20,
        output_dir = "outputs",
        optim = "adamw_8bit",
    ),
)

trainer.train()

Veri çekiliyor (Stooq)...
Veri işleniyor... (10x10 Çözünürlük)
Veri seti hazırlanıyor... Toplam gün sayısı: 2512
Llama-3 Modeli Yükleniyor (4-bit)...
==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map (num_proc=2):   0%|          | 0/2206 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,206 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,0.829000
40,0.804700
60,0.797000


TrainOutput(global_step=60, training_loss=0.8102203051249186, metrics={'train_runtime': 1715.4633, 'train_samples_per_second': 0.28, 'train_steps_per_second': 0.035, 'total_flos': 2.225661850681344e+16, 'train_loss': 0.8102203051249186, 'epoch': 0.21758839528558477})

In [ ]:
# --- HÜCRE 4: Değerlendirme (Inference & RMSE) ---
from sklearn.metrics import mean_squared_error
FastLanguageModel.for_inference(model)

predictions = []
actuals = []
test_data = train_test_split["test"]
eval_limit = min(100, len(test_data))

print(f"\n--- DEĞERLENDİRME BAŞLIYOR ({eval_limit} örnek) ---")

for i in range(eval_limit):
    full_text = test_data[i]["text"]
    prompt_text = full_text.split("\nResponse:")[0] + "\nResponse:"
    true_token = full_text.split("\nResponse: ")[1]

    inputs = tokenizer([prompt_text], return_tensors = "pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens = 5, use_cache = True)
    pred_text = tokenizer.batch_decode(outputs)[0]

    response_part = pred_text.split("\nResponse:")[-1].strip()
    pred_token = response_part.split(" ")[0]

    pred_val = token_to_value.get(pred_token, 0.0)
    true_val = token_to_value.get(true_token, 0.0)

    predictions.append(pred_val)
    actuals.append(true_val)

lstm_baseline_rmse = 0.01389
slm_rmse = np.sqrt(mean_squared_error(actuals, predictions))

print(f"\n=== NİHAİ SONUÇLAR (JUDGMENT DAY) ===")
print(f"LSTM Baseline RMSE : {lstm_baseline_rmse:.5f}")
print(f"SLM (Llama-3) RMSE : {slm_rmse:.5f}")

if slm_rmse < lstm_baseline_rmse:
    print("\n✅ BAŞARILI! Finwise Scribe (Sembolik SLM) hipotezi kanıtladı.")
else:
    print("\n❌ TEKRAR BAŞARISIZ. Hipotez geliştirilmeli.")

# --- KRİTİK GÜNCELLEME: SADECE ADAPTÖRÜ KAYDETME ---
# OOM hatasını engeller
print("\n--- LoRA Adaptörünü Güvenli Olarak Kaydetme ---")
from google.colab import drive
drive.mount('/content/drive')
import shutil
import os

ADAPTER_SAVE_DIR = "finwise_scribe_adapter"
ADAPTER_ZIP_PATH = "/content/drive/MyDrive/finwise_scribe_adapter_v1.zip"

model.save_pretrained(ADAPTER_SAVE_DIR)
tokenizer.save_pretrained(ADAPTER_SAVE_DIR)

!zip -r finwise_scribe_adapter.zip {ADAPTER_SAVE_DIR}

if os.path.exists("finwise_scribe_adapter.zip"):
    shutil.copy("finwise_scribe_adapter.zip", ADAPTER_ZIP_PATH)
    print(f"✅ LoRA Adaptör ZIP dosyası Drive'a kopyalandı: {ADAPTER_ZIP_PATH}")
    print("Artık bu ZIP dosyasını indirip lokal projenizde birleştirebilirsiniz.")
else:
    print("❌ Adaptör ZIP dosyası oluşturulamadı.")

Unsloth: Input IDs of shape torch.Size([1, 1816]) with length 1816 > the model's max sequence length of 1024.
We shall truncate it ourselves. It's imperative if you correct this issue first.



--- DEĞERLENDİRME BAŞLIYOR (100 örnek) ---

=== NİHAİ SONUÇLAR (JUDGMENT DAY) ===
LSTM Baseline RMSE : 0.01389
SLM (Llama-3) RMSE : 0.01657

❌ TEKRAR BAŞARISIZ. Hipotez geliştirilmeli.

--- LoRA Adaptörünü Güvenli Olarak Kaydetme ---
Mounted at /content/drive
  adding: finwise_scribe_adapter/ (stored 0%)
  adding: finwise_scribe_adapter/tokenizer_config.json (deflated 96%)
  adding: finwise_scribe_adapter/adapter_config.json (deflated 57%)
  adding: finwise_scribe_adapter/README.md (deflated 65%)
  adding: finwise_scribe_adapter/adapter_model.safetensors (deflated 7%)
  adding: finwise_scribe_adapter/special_tokens_map.json (deflated 71%)
  adding: finwise_scribe_adapter/tokenizer.json (deflated 85%)
✅ LoRA Adaptör ZIP dosyası Drive'a kopyalandı: /content/drive/MyDrive/finwise_scribe_adapter_v1.zip
Artık bu ZIP dosyasını indirip lokal projenizde birleştirebilirsiniz.
